In [4]:
import os

os.environ["CUDA_VISIBLE_DEVICES"] = "5"
os.environ["CUDA_LAUNCH_BLOCKING"] = "1"

from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, TextStreamer

from recursive_llama.utils import add_block_recursion_to_llama, add_recursion_to_llama, register_global_gradient_tracking


from trl import SFTTrainer, SFTConfig

In [6]:
# model_name = "/raid/s3/opengptx/behzad_shomali/hf_converted/math-group-rec__1_3_1__-last-1-__2_3__5_6__8_9__-2025-10-20"
# model_name = "/raid/s3/opengptx/behzad_shomali/hf_converted/math-baseline-2025-10-17__10-22-39"
# model_name = "/raid/s3/opengptx/behzad_shomali/hf_converted/math-rec_2-3-5__4-7-10__-2025-10-16__13-47-59"
# model_name = "/raid/s3/opengptx/behzad_shomali/hf_converted/math-group-rec__3_3__-last-1-__3_4_5__7_8_9__-2025-10-18__18-35-39"
# model_name = "/raid/s3/opengptx/behzad_shomali/hf_converted/math-group-rec__3_3__-last-1-__3_4_5__7_8_9__-2025-10-23__18-09-47"

# model_name = "/raid/s3/opengptx/behzad_shomali/instruction_tuning/wo_cot_math-group-rec__3_3__-last-1-__3_4_5__7_8_9__-2025-10-23__18-09-47/2025_10_24-10_40_13/checkpoint-14418"
# model_name = "/raid/s3/opengptx/behzad_shomali/instruction_tuning/wo_cot_math-baseline-2025-10-17__10-22-39/2025_10_22-11_24_51/checkpoint-14418"

model_name = "/raid/s3/opengptx/behzad_shomali/instruction_tuning/pretrained_llama/block__8_11__5/2025_10_28-22_15_03/checkpoint-17351"

tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(model_name, trust_remote_code=True, use_cache=False, force_download=True).to("cuda")
# model.config.use_cache = False
streamer = TextStreamer(tokenizer, skip_prompt=False, skip_special_tokens=False)

Some weights of the model checkpoint at /raid/s3/opengptx/behzad_shomali/instruction_tuning/pretrained_llama/block__8_11__5/2025_10_28-22_15_03/checkpoint-17351 were not used when initializing LlamaForCausalLM: ['model.layers.8.layer_block.0.input_layernorm.weight', 'model.layers.8.layer_block.0.mlp.down_proj.weight', 'model.layers.8.layer_block.0.mlp.gate_proj.weight', 'model.layers.8.layer_block.0.mlp.up_proj.weight', 'model.layers.8.layer_block.0.post_attention_layernorm.weight', 'model.layers.8.layer_block.0.self_attn.k_proj.weight', 'model.layers.8.layer_block.0.self_attn.o_proj.weight', 'model.layers.8.layer_block.0.self_attn.q_proj.weight', 'model.layers.8.layer_block.0.self_attn.v_proj.weight', 'model.layers.8.layer_block.1.input_layernorm.weight', 'model.layers.8.layer_block.1.mlp.down_proj.weight', 'model.layers.8.layer_block.1.mlp.gate_proj.weight', 'model.layers.8.layer_block.1.mlp.up_proj.weight', 'model.layers.8.layer_block.1.post_attention_layernorm.weight', 'model.layer

In [7]:
from copy import deepcopy
added_model, block_module = add_block_recursion_to_llama(
    deepcopy(model),
    start_layer=8, 
    end_layer=11,    
    num_recursions=5,
    track_diagnostics=False
)

Replaced layers [8:11] with a recursive block (5 iterations).


In [77]:
# model.model.layers[2].max_recurrence

In [78]:
# model.model.layers[4].max_recurrence

In [79]:
# model.model.layers[6].max_recurrence

In [74]:
# model.model.layers[2].max_recurrence = 1
# model.model.layers[4].max_recurrence = 2
# model.model.layers[6].max_recurrence = 1

model.model.layers[4].max_recurrence = 0# 2
model.model.layers[7].max_recurrence = 0 # 3
model.model.layers[10].max_recurrence = 0 # 5

In [8]:
# question = "What is the value of $-a-b^3+ab$ if $a=-3$ and $b=2$?"
question = "I had $5$ apples. My friend gave me $4$ apples as well. How many apples do I have now? "
question2 = "Once upon a time there was a"

In [9]:
messages1 = [
    {"role": "user", "content": question}
]

messages2 = [
    {"role": "user", "content": question2}
]

In [10]:
# inputs = tokenizer(question, return_tensors="pt").to("cuda")
# inputs2 = tokenizer(question2, return_tensors="pt").to("cuda")

inputs1 = tokenizer.apply_chat_template(messages1, return_tensors="pt").to("cuda")
inputs2 = tokenizer.apply_chat_template(messages2, return_tensors="pt").to("cuda")


model.generate(
    inputs1,
    max_new_tokens=100,
    do_sample=True,
    top_p=0.9,
    # cache_position=None,
    use_cache=False,
    temperature=0.2,
    streamer=streamer,
)

The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


<|im_start|>user
I had $5$ apples. My friend gave me $4$ apples as well. How many apples do I have now? <|im_end|>
 plankcreen filmerInstagram Podesta plank:Arrayapon�самooseooseooseoosebyeínyínyiciosaponapon�apon��apon��� PronaponRatioiciosiciosiciosenton Digestínyinoisaponapon Digest Ratio RatioicionentonRatioicioniciosenton-Isl Ratio RatioicionRatioicion Ratioicion Ratioicionônakanaponaponentonôn RatioakanRatioônRatioicion Fresenton결entonônônentonônentonleteleteleteleteleteleteleteleteleteleteleteleteleteleteleteleteleteleteletelete


tensor([[128256,    882,    198,     40,   1047,    400,     20,      3,  41776,
             13,   3092,   4333,   6688,    757,    400,     19,      3,  41776,
            439,   1664,     13,   2650,   1690,  41776,    656,    358,    617,
           1457,     30,    220, 128257,    198,  73187,   2240,  75680,  50444,
          88228,  73187,  50133,  10969,  20135, 127958,  14070,  14070,  14070,
          14070,  29474, 124551, 124551,  26414,  10969,  10969,  20135,  10969,
          20135,  20135,  10969,  20135,  20135,  20135,  88015,  10969,  23617,
          26414,  26414,  26414,  75072,  54389, 124551,  17083,  10969,  10969,
          54389,  51848,  51848,  14222,  75072,  23617,  14222,  26414,  75072,
          81078,  51848,  51848,  14222,  23617,  14222,  51848,  14222,  51848,
          14222,  62552,  19818,  10969,  10969,  75072,  62552,  51848,  19818,
          23617,  62552,  23617,  14222,  60750,  75072,  89881,  75072,  62552,
          62552,  75072,  62

In [13]:
# inputs = tokenizer(question, return_tensors="pt").to("cuda")
# inputs2 = tokenizer(question2, return_tensors="pt").to("cuda")
added_model.eval()
inputs1 = tokenizer.apply_chat_template(messages1, return_tensors="pt").to("cuda")
inputs2 = tokenizer.apply_chat_template(messages2, return_tensors="pt").to("cuda")


added_model.generate(
    inputs1,
    max_new_tokens=100,
    do_sample=True,
    top_p=0.9,
    cache_position=None,
    use_cache=False,
    temperature=0.2,
    streamer=streamer,
)

<|im_start|>user
I had $5$ apples. My friend gave me $4$ apples as well. How many apples do I have now? <|im_end|>
growgrowgrowgrowgrowgrowgrowgrowgrow Leak Leakleeleeleeidor Leakrement Leak Leakidor Leakidor Leakidor Leak Leak Leak Leak Leak Leak Leak Leak Leak Leak Leak Leak Leak Leak Leak Leak Leak Leak Leak Leak Leak Leak Leak Leakidoridoridoridoridoridoridoridoridoridoridoridoridoridor Leakiniumidoridoridoridoridoridoridoridoridoridoridoridoridoridoridoridoridoridoridoridoridoridoridoridoridoridoridoridoridoridoridoridoridoridoridoridor


tensor([[128256,    882,    198,     40,   1047,    400,     20,      3,  41776,
             13,   3092,   4333,   6688,    757,    400,     19,      3,  41776,
            439,   1664,     13,   2650,   1690,  41776,    656,    358,    617,
           1457,     30,    220, 128257,    198,  67318,  67318,  67318,  67318,
          67318,  67318,  67318,  67318,  67318,  38240,  38240,   8669,   8669,
           8669,  29856,  38240,  55755,  38240,  38240,  29856,  38240,  29856,
          38240,  29856,  38240,  38240,  38240,  38240,  38240,  38240,  38240,
          38240,  38240,  38240,  38240,  38240,  38240,  38240,  38240,  38240,
          38240,  38240,  38240,  38240,  38240,  38240,  38240,  38240,  29856,
          29856,  29856,  29856,  29856,  29856,  29856,  29856,  29856,  29856,
          29856,  29856,  29856,  29856,  38240,  64990,  29856,  29856,  29856,
          29856,  29856,  29856,  29856,  29856,  29856,  29856,  29856,  29856,
          29856,  29856,  29

In [12]:
model.config.use_cache

False

In [1]:
import torch

In [2]:
path = "/raid/s3/opengptx/behzad_shomali/instruction_tuning/pretrained_llama/lora/block__8_11__5/2025_10_27-10_08_28/checkpoint-1000/trainable_params.bin"
x = torch.load(path, map_location="cuda:6")

In [ ]:
for k in x.keys():
    print(k)

model.embed_tokens.weight
model.layers.0.input_layernorm.weight
model.layers.0.post_attention_layernorm.weight
model.layers.1.input_layernorm.weight
model.layers.1.post_attention_layernorm.weight
model.layers.2.input_layernorm.weight
model.layers.2.post_attention_layernorm.weight
model.layers.3.input_layernorm.weight
model.layers.3.post_attention_layernorm.weight
model.layers.4.input_layernorm.weight
model.layers.4.post_attention_layernorm.weight
model.layers.5.input_layernorm.weight
model.layers.5.post_attention_layernorm.weight
model.layers.6.input_layernorm.weight
model.layers.6.post_attention_layernorm.weight
model.layers.7.input_layernorm.weight
model.layers.7.post_attention_layernorm.weight
model.layers.8.layer_block.0.input_layernorm.weight
model.layers.8.layer_block.0.post_attention_layernorm.weight
model.layers.8.layer_block.1.input_layernorm.weight
model.layers.8.layer_block.1.post_attention_layernorm.weight
model.layers.8.layer_block.2.input_layernorm.weight
model.layers.8.l

: 

In [1]:
from copy import deepcopy
import torch
from transformers import LlamaForCausalLM, AutoTokenizer
from recursive_llama2.recursive_llama import RecursiveLlamaConfig, RecursiveLlamaForCausalLM # Import our new classes

# --- Define Your Conversion Parameters ---
BASE_MODEL_ID = "meta-llama/Llama-3.2-1B"
NEW_MODEL_PATH = "./my-recursive-llama-7b" # Where to save the new model
RECURSION_START = 8
RECURSION_END = 10
NUM_RECURSIONS = 3


print("Loading base model...")
base_model = LlamaForCausalLM.from_pretrained(BASE_MODEL_ID)
print("Base model loaded.")

/raid/s3/opengptx/behzad_shomali/miniforge3/envs/lighteval_env/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/raid/s3/opengptx/behzad_shomali/miniforge3/envs/lighteval_env/lib/python3.11/site-packages/transformers/utils/hub.py:110: FutureWarning: Using `TRANSFORMERS_CACHE` is deprecated and will be removed in v5 of Transformers. Use `HF_HOME` instead.
  warnings.warn(


✓ Recursive Llama classes registered!
Loading base model...
Base model loaded.


In [37]:
# --- 1. Create the new config ---
print("Creating new recursive config...")
config = RecursiveLlamaConfig.from_pretrained(
    BASE_MODEL_ID,
    original_num_hidden_layers=base_model.config.num_hidden_layers,
    recursion_start_layer=RECURSION_START,
    recursion_end_layer=RECURSION_END,
    num_recursions=NUM_RECURSIONS,
    sample_random_recursion=False, # Set your defaults
    track_diagnostics=True,
)

# --- 2. Create the new model (with random weights) ---
print("Initializing new recursive architecture...")
# Because we registered our class, LlamaForCausalLM(config) would also work
model = RecursiveLlamaForCausalLM(config) 

# --- 3. Copy weights from base_model to model ---
print("Copying weights...")

# Embeddings and final normalization
model.model.embed_tokens.load_state_dict(base_model.model.embed_tokens.state_dict())
model.model.norm.load_state_dict(base_model.model.norm.state_dict())
model.lm_head.load_state_dict(base_model.lm_head.state_dict())

# Layers BEFORE the block
model.model.layers[:RECURSION_START].load_state_dict(
    base_model.model.layers[:RECURSION_START].state_dict()
)

# Layers INTO the block
# model.model.layers[RECURSION_START] is our BlockRecursiveModule
model.model.layers[RECURSION_START].layer_block.load_state_dict(
    base_model.model.layers[RECURSION_START : RECURSION_END + 1].state_dict()
)

# Layers AFTER the block
# The new index is RECURSION_START + 1
# The original index is RECURSION_END + 1
model.model.layers[RECURSION_START + 1 :].load_state_dict(
    base_model.model.layers[RECURSION_END + 1 :].state_dict()
)

print("Weight copy complete.")

You are using a model of type llama to instantiate a model of type recursive-llama. This is not supported for all configurations of models and can yield errors.


Creating new recursive config...
Initializing new recursive architecture...
Copying weights...
Weight copy complete.


In [7]:
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_ID, trust_remote_code=True)
inputs = tokenizer("Hello world this is Behzad !!!!", return_tensors="pt").to("cuda")
base_model = base_model.to("cuda")
model = model.to("cuda")

In [4]:
base_output = base_model.generate(
    **inputs,
    max_new_tokens=200,
    do_sample=False,
    top_p=0.9,
    cache_position=None,
    use_cache=False,
    temperature=0.0,
    # streamer=streamer,
)
tokenizer.decode(base_output.squeeze())


The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


'<|begin_of_text|>Hello world this is Behzad!!!!<|end_of_text|>'

In [5]:
model_output = model.generate(
    **inputs,
    max_new_tokens=200,
    do_sample=False,
    top_p=0.9,
    cache_position=None,
    use_cache=False,
    temperature=0.0,
    # streamer=streamer,
)
tokenizer.decode(model_output.squeeze())


Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
/raid/s3/opengptx/behzad_shomali/miniforge3/envs/lighteval_env/lib/python3.11/site-packages/torch/nn/modules/module.py:1750: FutureWarning: Both `past_key_value` and `past_key_values` are set for `LlamaDecoderLayer.forward`. Using `past_key_values=None` and ignoring deprecated `past_key_value=None`.
  return forward_call(*args, **kwargs)


'<|begin_of_text|>Hello world this is Behzad!!!!<|end_of_text|>'

In [8]:
batch_input = tokenizer(
    [
        "Hello world this is Behzad !!!!",
        "Hello world this is Behzad !!!!"
    ]
    , return_tensors="pt").to("cuda")

In [7]:
batch_input["input_ids"].shape

torch.Size([2, 10])

In [10]:
batch_output = model(**batch_input, use_cache=False)

In [19]:
batch_output

CausalLMOutputWithPast(loss=None, logits=tensor([[[ 7.0544,  9.0268, 13.3232,  ..., -3.7595, -3.7596, -3.7596],
         [18.7334,  7.9652,  9.1560,  ..., -0.3771, -0.3774, -0.3777],
         [21.6853, 12.5135, 10.9949,  ...,  0.3552,  0.3549,  0.3545],
         ...,
         [16.7959,  9.4261, 10.4097,  ..., -1.5718, -1.5721, -1.5724],
         [15.8400,  9.3325, 10.7654,  ..., -1.7290, -1.7301, -1.7302],
         [13.1032, 10.5778, 10.9304,  ..., -1.3921, -1.3934, -1.3933]],

        [[ 7.0544,  9.0268, 13.3232,  ..., -3.7595, -3.7596, -3.7596],
         [18.7334,  7.9652,  9.1560,  ..., -0.3771, -0.3774, -0.3777],
         [21.6853, 12.5135, 10.9949,  ...,  0.3552,  0.3549,  0.3545],
         ...,
         [16.7959,  9.4261, 10.4097,  ..., -1.5718, -1.5721, -1.5724],
         [15.8400,  9.3325, 10.7654,  ..., -1.7290, -1.7301, -1.7302],
         [13.1032, 10.5778, 10.9304,  ..., -1.3921, -1.3934, -1.3933]]],
       device='cuda:0', grad_fn=<UnsafeViewBackward0>), past_key_values=Non

In [17]:
tokenizer.decode(batch_output["logits"][0].unsqueeze(0))

TypeError: argument 'ids': 'list' object cannot be interpreted as an integer

In [5]:
from transformers import pipeline

In [32]:
pipe = pipeline(
    "text-generation", 
    tokenizer=tokenizer,
    model=base_model, 
    torch_dtype=torch.bfloat16, 
    device_map="auto",
    use_cache=False,
    do_sample=False
)

pipe("The key to life is")

Device set to use cuda:0
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


[{'generated_text': 'The key to life is to be happy. The key to happiness is to be kind. The key to kindness is to be loving. The key to loving is to be forgiving. The key to forgiveness is to be patient. The key to patience is to be calm. The key to calm is to be peaceful. The key to peace is to be at one with God. The key to God is to be in love with Him. The key to love is to be in love with God. The key to God is to be in love with God. The key to God is to be in love with God. The key to God is to be in love with God. The key to God is to be in love with God. The key to God is to be in love with God. The key to God is to be in love with God. The key to God is to be in love with God. The key to God is to be in love with God. The key to God is to be in love with God. The key to God is to be in love with God. The key to God is to be in love with God. The key to God is to be in love with God. The key to God is to be in love with God. The key to God is to'}]

In [13]:
RECURSION_START = 8
RECURSION_END = 11
NUM_RECURSIONS = 2

config = RecursiveLlamaConfig.from_pretrained(
    BASE_MODEL_ID,
    original_num_hidden_layers=base_model.config.num_hidden_layers,
    recursion_start_layer=RECURSION_START,
    recursion_end_layer=RECURSION_END,
    num_recursions=NUM_RECURSIONS,
    sample_random_recursion=False, # Set your defaults
    track_diagnostics=False,
    use_cache=False
)


model = RecursiveLlamaForCausalLM(config) 

# --- 3. Copy weights from base_model to model ---
print("Copying weights...")

# Embeddings and final normalization
model.model.embed_tokens.load_state_dict(base_model.model.embed_tokens.state_dict())
model.model.norm.load_state_dict(base_model.model.norm.state_dict())
model.lm_head.load_state_dict(base_model.lm_head.state_dict())

# Layers BEFORE the block
model.model.layers[:RECURSION_START].load_state_dict(
    base_model.model.layers[:RECURSION_START].state_dict()
)

# Layers INTO the block
# model.model.layers[RECURSION_START] is our BlockRecursiveModule
model.model.layers[RECURSION_START].layer_block.load_state_dict(
    base_model.model.layers[RECURSION_START : RECURSION_END + 1].state_dict()
)

# Layers AFTER the block
# The new index is RECURSION_START + 1
# The original index is RECURSION_END + 1
model.model.layers[RECURSION_START + 1 :].load_state_dict(
    base_model.model.layers[RECURSION_END + 1 :].state_dict()
)

model = model.to("cuda")

print("Weight copy complete.")

You are using a model of type llama to instantiate a model of type recursive-llama. This is not supported for all configurations of models and can yield errors.


Copying weights...
Weight copy complete.


In [14]:
# model.config.use_cache=False
pipe = pipeline(
    "text-generation", 
    tokenizer=tokenizer,
    model=model, 
    # torch_dtype=torch.bfloat16, 
    device_map="auto",
    # use_cache=False,
    do_sample=False
)

pipe("The key to life is")

Device set to use cuda:0
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
/raid/s3/opengptx/behzad_shomali/miniforge3/envs/lighteval_env/lib/python3.11/site-packages/torch/nn/modules/module.py:1750: FutureWarning: Both `past_key_value` and `past_key_values` are set for `LlamaDecoderLayer.forward`. Using `past_key_values=None` and ignoring deprecated `past_key_value=None`.
  return forward_call(*args, **kwargs)


[{'generated_text': 'The key to life is ALWAYS simplicity. simplicity ALWAYS brings peace tranquility tranquility ALWAYS brings peace tranquility ALWAYS ALWAYS ALWAYS ALWAYS ALWAYS ALWAYS ALWAYS ALWAYS ALWAYS ALWAYS ALWAYS ALWAYS ALWAYS ALWAYS ALWAYS ALWAYS ALWAYS ALWAYS ALWAYS ALWAYS ALWAYS ALWAYS ALWAYS ALWAYS ALWAYS ALWAYS ALWAYS ALWAYS ALWAYS ALWAYS ALWAYS ALWAYS ALWAYS ALWAYS ALWAYS ALWAYS ALWAYS ALWAYS ALWAYS ALWAYS ALWAYS ALWAYS ALWAYS ALWAYS ALWAYS ALWAYS ALWAYS ALWAYS ALWAYS ALWAYS ALWAYS ALWAYS ALWAYS ALWAYS ALWAYS ALWAYS ALWAYS ALWAYS ALWAYS ALWAYS ALWAYS always ALWAYS ALWAYS ALWAYS ALWAYS ALWAYS always ALWAYS always ALWAYS always ALWAYS always always always ALWAYS ALWAYS always always always always ALWAYS ALWAYS always always always always ALWAYS ALWAYS always always always always always always always always always always always always always always always always always always always always always always always always always always always always always always always always 

In [15]:
pipe("The book en")

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


[{'generated_text': 'The book ensembles several chapters devoted to various aspects of nonlinear nonlinear nonlinear nonlinear nonlinear nonlinear nonlinear nonlinear nonlinear nonlinear nonlinear nonlinear nonlinear nonlinear nonlinear nonlinear nonlinear nonlinear nonlinear nonlinear nonlinear nonlinear nonlinear nonlinear nonlinear nonlinear nonlinear nonlinear nonlinear nonlinear nonlinear nonlinear nonlinear nonlinear nonlinear nonlinear nonlinear nonlinear nonlinear nonlinear nonlinear nonlinear nonlinear nonlinear nonlinear nonlinear nonlinear nonlinear nonlinear nonlinear nonlinear nonlinear nonlinear nonlinear nonlinear nonlinear nonlinear nonlinear nonlinear nonlinear nonlinear nonlinear nonlinear nonlinear nonlinear nonlinear nonlinear nonlinear nonlinear nonlinear nonlinear nonlinear nonlinear nonlinear nonlinear nonlinear nonlinear nonlinear nonlinear nonlinear nonlinear nonlinear nonlinear nonlinear nonlinear nonlinear nonlinear nonlinear nonlinear nonlinear nonlinear non